# Continual Learning — Colab Training Notebook

**Before running:**
1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU or better)
2. Edit the **Configuration** cell below
3. Run all cells top-to-bottom

Checkpoints and the dataset cache are stored on **Google Drive** and survive session restarts.
If a checkpoint already exists for your `RUN_ID`, training will automatically resume from it.

## Configuration
Edit the values below before running the notebook.

In [16]:
# ============================================================
#  CONFIGURE EVERYTHING HERE
# ============================================================

# Training method — choose one:
#   "full_ft"  — fine-tune all parameters (catastrophic forgetting baseline)
#   "lora"     — parameter-efficient LoRA adapters
#   "smf"      — frozen backbone + sparse trainable memory
#   "casm"     — versioned memory bank with contradiction-aware routing
METHOD = "full_ft"

# HuggingFace model (must have approved access)
MODEL_NAME = "meta-llama/Llama-3.2-1B"

# Unique experiment name — checkpoints are stored under this ID
RUN_ID = "llama1b_run_01"

# Periods to train — full sequence is all four in order
# Each period is one temporal snapshot of Wikipedia (~2021)
PERIODS = ["aug_sep", "sep_oct"]  # add "oct_nov", "nov_dec" for full run

# Google Drive folder — checkpoints and dataset cache go here
DRIVE_DIR = "/content/drive/MyDrive/continual_learning"

# ---- Core hyperparameters ----
BATCH_SIZE             = 1
GRAD_ACCUM_STEPS       = 8      # effective batch size = BATCH_SIZE * GRAD_ACCUM_STEPS
LEARNING_RATE          = 2e-4
EPOCHS_PER_PERIOD      = 1
MAX_PASSAGES_PER_PERIOD = 200   # set to None to use all passages in each period
PRECISION              = "bfloat16"  # "bfloat16" | "float16"
SEED                   = 42

# ---- Activation capture ----
CAPTURE_ACTIVATIONS = True

# ---- LoRA settings (only used when METHOD == "lora") ----
LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

# ---- SMF settings (only used when METHOD == "smf") ----
# Llama-3.2-1B has 16 transformer layers (indices 0-15)
SMF_MEMORY_SIZE            = 64
SMF_SPARSITY_RATIO         = 0.1
SMF_UPDATE_LAYERS          = [4, 8, 12]  # mid-to-late layers
SMF_REGULARIZATION_WEIGHT  = 0.01
SMF_LEARNING_RATE          = 1e-3        # higher LR for the small memory module

# ---- CASM settings (only used when METHOD == "casm") ----
CASM_NUM_SLOTS              = 8
CASM_ROUTER_HIDDEN_SIZE     = 256
CASM_TOP_K                  = 2
CASM_ROUTER_TEMPERATURE     = 1.0
CASM_SPARSITY_WEIGHT        = 0.01
CASM_OVERLAP_WEIGHT         = 0.01
CASM_BRANCH_ON_CONTRADICTION = True
CASM_MEMORY_SIZE            = 64

## Setup

In [17]:
# Mount Google Drive for persistent checkpoint and dataset storage
from google.colab import drive
drive.mount("/content/drive")

import os

CHECKPOINT_DIR   = os.path.join(DRIVE_DIR, "checkpoints")
DATASET_CACHE_DIR = os.path.join(DRIVE_DIR, "dataset_cache")
REPO_DIR         = "/content/Continual-Learning"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DATASET_CACHE_DIR, exist_ok=True)

print(f"Checkpoint dir:  {CHECKPOINT_DIR}")
print(f"Dataset cache:   {DATASET_CACHE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint dir:  /content/drive/MyDrive/continual_learning/checkpoints
Dataset cache:   /content/drive/MyDrive/continual_learning/dataset_cache


In [18]:
# Clone the repo (or pull if already cloned this session)
import subprocess

REPO_URL = "https://github.com/athirai-s/Continual-Learning"

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    print("Repo already cloned — pulling latest changes...")
    result = subprocess.run(
        ["git", "-C", REPO_DIR, "pull"],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or result.stderr.strip())
else:
    print(f"Cloning {REPO_URL} ...")
    result = subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{result.stderr}")

print("Repo ready.")

Repo already cloned — pulling latest changes...
Already up to date.
Repo ready.


In [19]:
# Install dependencies not pre-installed on Colab
# transformers 5.x is required — Colab ships with 4.x by default
!pip install -q "transformers>=5.3.0" "peft>=0.18.1" "trl>=0.29.0" "datasets>=4.6.1"

# Add the repo to the Python path so imports work
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Dependencies installed. Repo on path.")

Dependencies installed. Repo on path.


## Dataset

The TemporalWiki dataset is downloaded once to Google Drive and reused across sessions.
Each session the zips are copied from Drive into the repo's `data/` folder where the
dataset loader expects them.

In [20]:
import shutil
import requests
from tqdm.auto import tqdm

HF_BASE = "https://huggingface.co/datasets/seonghyeonye/TemporalWiki/resolve/main"
DATASET_FILES = ["TWiki_Probes.zip", "TWiki_Diffsets.zip"]

def _download_file(url: str, dest: str) -> None:
    """Stream-download url → dest with a progress bar."""
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    resp = requests.get(url, stream=True)
    resp.raise_for_status()
    total = int(resp.headers.get("content-length", 0))
    with open(dest, "wb") as f, tqdm(
        desc=os.path.basename(dest), total=total, unit="B", unit_scale=True
    ) as bar:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
            bar.update(len(chunk))

data_dir = os.path.join(REPO_DIR, "data")
os.makedirs(data_dir, exist_ok=True)

for filename in DATASET_FILES:
    cache_path = os.path.join(DATASET_CACHE_DIR, filename)
    repo_path  = os.path.join(data_dir, filename)

    # Download to Drive cache only if not already cached
    if not os.path.exists(cache_path):
        print(f"Downloading {filename} → Drive cache (one-time)...")
        _download_file(f"{HF_BASE}/{filename}", cache_path)
    else:
        print(f"{filename}: found in Drive cache.")

    # Copy from Drive cache into repo/data/ for this session
    if not os.path.exists(repo_path):
        print(f"Copying {filename} to repo/data/ ...")
        shutil.copy2(cache_path, repo_path)
    else:
        print(f"{filename}: already in repo/data/.")

print("\nDataset ready.")

TWiki_Probes.zip: found in Drive cache.
TWiki_Probes.zip: already in repo/data/.
TWiki_Diffsets.zip: found in Drive cache.
TWiki_Diffsets.zip: already in repo/data/.

Dataset ready.


## HuggingFace Authentication

Required for gated models like Llama. Log in with a token that has **read** access to
`meta-llama/Llama-3.2-1B`. You can create a token at https://huggingface.co/settings/tokens

In [21]:
from huggingface_hub import login
login()  # will prompt for your HuggingFace access token

## Training

In [22]:
# Build TrainConfig from configuration variables set at the top
from training.train_config import TrainConfig

config_kwargs = dict(
    run_id=RUN_ID,
    model_name=MODEL_NAME,
    method=METHOD,
    dataset_name="temporal_wiki",
    precision=PRECISION,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    epochs_per_period=EPOCHS_PER_PERIOD,
    max_passages_per_period=MAX_PASSAGES_PER_PERIOD,
    log_every_n_steps=10,
    capture_activations=CAPTURE_ACTIVATIONS,
    eval_after_each_period=True,
    seed=SEED,
)

if METHOD == "lora":
    config_kwargs.update(
        lora_r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        lora_target_modules=LORA_TARGET_MODULES,
    )
elif METHOD == "smf":
    config_kwargs.update(
        smf_memory_size=SMF_MEMORY_SIZE,
        smf_sparsity_ratio=SMF_SPARSITY_RATIO,
        smf_update_layers=SMF_UPDATE_LAYERS,
        smf_regularization_weight=SMF_REGULARIZATION_WEIGHT,
        smf_learning_rate=SMF_LEARNING_RATE,
        smf_freeze_backbone=True,
    )
elif METHOD == "casm":
    config_kwargs.update(
        casm_num_slots=CASM_NUM_SLOTS,
        casm_router_hidden_size=CASM_ROUTER_HIDDEN_SIZE,
        casm_top_k=CASM_TOP_K,
        casm_router_temperature=CASM_ROUTER_TEMPERATURE,
        casm_sparsity_weight=CASM_SPARSITY_WEIGHT,
        casm_overlap_weight=CASM_OVERLAP_WEIGHT,
        casm_branch_on_contradiction=CASM_BRANCH_ON_CONTRADICTION,
        casm_memory_size=CASM_MEMORY_SIZE,
    )

cfg = TrainConfig(**config_kwargs)
cfg.validate()

print(f"Method:             {cfg.method}")
print(f"Model:              {cfg.model_name}")
print(f"Periods:            {PERIODS}")
print(f"Batch size:         {cfg.batch_size}")
print(f"Grad accum steps:   {cfg.grad_accum_steps}  (eff. batch = {cfg.batch_size * cfg.grad_accum_steps})")
print(f"Learning rate:      {cfg.learning_rate}")
print(f"Epochs per period:  {cfg.epochs_per_period}")
print(f"Max passages:       {cfg.max_passages_per_period}")
print(f"Checkpoint dir:     {CHECKPOINT_DIR}")

Method:             full_ft
Model:              meta-llama/Llama-3.2-1B
Periods:            ['aug_sep', 'sep_oct']
Batch size:         1
Grad accum steps:   8  (eff. batch = 8)
Learning rate:      0.0002
Epochs per period:  1
Max passages:       200
Checkpoint dir:     /content/drive/MyDrive/continual_learning/checkpoints


In [23]:
# Detect existing checkpoint for automatic resume
import json

RESUME_FROM = None
run_root = os.path.join(CHECKPOINT_DIR, RUN_ID)
latest_json = os.path.join(run_root, "latest.json")

if os.path.exists(latest_json):
    with open(latest_json) as f:
        pointer = json.load(f)
    last_period = pointer.get("last_period", "unknown")
    print(f"Existing checkpoint found (last period: {last_period}).")
    print(f"Training will resume from: {run_root}")
    RESUME_FROM = run_root
else:
    print("No existing checkpoint — starting fresh.")

No existing checkpoint — starting fresh.


In [24]:
# Run training
# Uses run_training() directly so we can pass the explicit PERIODS list.
from training.train_runner import (
    run_training,
    build_real_model_and_tokenizer,
    load_real_model_and_tokenizer,
    build_real_dataset,
)

results = run_training(
    cfg,
    model_factory=build_real_model_and_tokenizer,
    resume_model_factory=load_real_model_and_tokenizer,
    dataset_factory=build_real_dataset,
    checkpoint_dir=CHECKPOINT_DIR,
    training_units=PERIODS,
    resume_from=RESUME_FROM,
)

print("\n" + "="*50)
print("TRAINING SUMMARY")
print("="*50)
for i, r in enumerate(results):
    period = PERIODS[i] if i < len(PERIODS) else f"period_{i}"
    print(f"\n{period}:")
    loss = r.get("train_loss_final")
    print(f"  Final loss:        {loss:.4f}" if loss is not None else "  Final loss:        N/A")
    print(f"  Passages trained:  {r.get('n_passages_trained', 'N/A')}")
    print(f"  Train time (s):    {r.get('train_duration_sec', 0):.1f}")
    if "evaluation" in r:
        for split, eval_result in r["evaluation"].items():
            print(
                f"  Eval [{split:9s}]: "
                f"plasticity={eval_result.plasticity:.3f}  "
                f"stability={eval_result.stability:.3f}  "
                f"token_f1={eval_result.token_f1:.3f}"
            )
    print(f"  Checkpoint:        {r.get('checkpoint_path', 'N/A')}")

Training config:
TrainConfig(model_name='meta-llama/Llama-3.2-1B', method='full_ft', dataset_name='temporal_wiki', precision='bfloat16', learning_rate=0.0002, batch_size=1, grad_accum_steps=8, epochs_per_period=1, grad_clip=1.0, warmup_steps=100, min_passage_length=100, deduplicate=True, weighted_sampling=False, max_passages_per_period=200, shuffle_by_relation=True, contradiction_batch_frac=0.25, run_id='llama1b_run_01', log_every_n_steps=10, eval_after_each_period=True, capture_activations=False, checkpoint_dir='/content/drive/MyDrive/continual_learning/checkpoints', checkpoint_every_n_optimizer_steps=None, seed=42, lora_r=None, lora_alpha=None, lora_dropout=0.05, lora_target_modules=None, smf_memory_size=None, smf_sparsity_ratio=None, smf_update_layers=None, smf_regularization_weight=0.01, smf_freeze_backbone=True, smf_learning_rate=None, casm_num_slots=None, casm_router_hidden_size=None, casm_top_k=None, casm_router_temperature=1.0, casm_sparsity_weight=0.0, casm_overlap_weight=0.0,

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Saved config to /content/drive/MyDrive/continual_learning/checkpoints/llama1b_run_01_config.json


=== Training unit: aug_sep ===
  Columns in twiki_probes/0801-0901_changed.csv: ['subject', 'relation', 'object']
  Sample row:
subject     Nagpur Metro Rail
relation             has part
object         Congress Nagar
Name: 0, dtype: object

  Columns in twiki_probes/0801-0901_unchanged.csv: ['subject', 'relation', 'object']
  Sample row:
subject            Zola
relation    instance of
object         business
Name: 0, dtype: object

Using device: cuda
Training on 200 ordered passages


KeyboardInterrupt: 